# Parallelization helpers (MPI)

Use MPI to split work and reduce results:

In [1]:
using MatsubaraFunctions
using MPI

MPI.Init()
comm = mpi_comm()
rank = mpi_rank()
size = mpi_size()
println("Hello from rank $rank out of $size processes")

Hello from rank 0 out of 1 processes


In [10]:
chunk = mpi_split(1:100)          # rank-local subrange
local_sum = sum(chunk)

5050

Reduce MeshFunction data in-place (here, we see nothing special, because we run just on one process):


In [9]:
T = 0.1
N = 5
mf_fermi = MatsubaraMesh(T, N, Fermion)
Gω = MeshFunction(mf_fermi; data_t=Float64)

# Fill Gω.data with something rank-dependent
Gω.data .= MPI.Comm_rank(MPI.COMM_WORLD) + 1.0

# Perform the allreduce sum across all ranks
mpi_allreduce!(Gω)

# Print the result on each rank
println("Rank $rank has data: $(Gω.data)")

Rank 0 has data: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [11]:
MPI.Finalize()

To make use of MPI parallelization, run with:
```sh
mpiexec -n 4 julia --project -e 'using Pkg; Pkg.instantiate(); include("your_script.jl")'
```